# 🧬 Two-Objective qLogEHVI Campaign

This notebook demonstrates the two-objective qLogEHVI workflow introduced in v1.1.0 and still supported in v1.1.1. Every observed row records both objective values, and qLogEHVI suggests candidates against the configured Pareto reference point.

In [ ]:
from pathlib import Path
from shutil import copyfile
import os
import sys

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

mpl_cache = PROJECT_ROOT / ".matplotlib-cache"
mpl_cache.mkdir(exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(mpl_cache))

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from bo_forge.session import CampaignSession

CONFIG_PATH = PROJECT_ROOT / "configs/10_multi_objective_mixed_constrained_qlogehvi.yaml"
SEED_LOG_PATH = PROJECT_ROOT / "examples/10_multi_objective_mixed_constrained_campaign_log.csv"
WORKING_LOG_PATH = PROJECT_ROOT / "examples/10_multi_objective_mixed_constrained_working_log.csv"
LATEST_SUGGESTIONS_PATH = PROJECT_ROOT / "examples/10_multi_objective_mixed_constrained_latest_suggestions.csv"
REPORTS_DIR = PROJECT_ROOT / "reports"
REPORT_PATH = REPORTS_DIR / "10_multi_objective_campaign_report.txt"
PARETO_PLOT_PATH = REPORTS_DIR / "10_multi_objective_pareto.pdf"
HYPERVOLUME_PLOT_PATH = REPORTS_DIR / "10_multi_objective_hypervolume.pdf"
TARGET_OBSERVED_ROWS = 15

copyfile(SEED_LOG_PATH, WORKING_LOG_PATH)

campaign = CampaignSession.from_files(CONFIG_PATH, WORKING_LOG_PATH)
campaign.validate()
campaign.summary()

## 📍 Coupled Objective State

The campaign has two objectives: maximise `yield_score` and minimise `waste_score`. The reference point is specified in user-facing units and should be worse than the region of interest.

In [ ]:
campaign.next_action()

In [ ]:
campaign.pareto_summary()

In [ ]:
campaign.pareto_front()[["row_id", "yield_score", "waste_score"]]

## 🧪 Synthetic Coupled Evaluation

This simulator returns both objective values for the same experiment. v1.1.1 still assumes coupled objective evaluation and does not support missing or asynchronous objective values.

In [ ]:
def simulate_coupled_objectives(row: pd.Series) -> dict[str, float]:
    loading = float(row["catalyst_loading"])
    time = float(row["reaction_time"])
    base = float(row["base_equivalents"])
    solvent = str(row["solvent"])

    loading_scaled = (loading - 0.02) / 0.18
    time_scaled = (time - 20.0) / 70.0
    base_penalty = abs(base - 1.0)
    solvent_yield_bonus = {"MeCN": 4.0, "DMF": 1.5, "Water": -2.0}[solvent]
    solvent_waste = {"MeCN": 5.0, "DMF": 3.0, "Water": 1.0}[solvent]

    yield_peak = np.exp(-0.5 * (((loading_scaled - 0.62) / 0.18) ** 2 + ((time_scaled - 0.68) / 0.22) ** 2))
    yield_score = 42.0 + 35.0 * yield_peak - 7.0 * base_penalty + solvent_yield_bonus

    waste_score = 9.0 + 16.0 * loading_scaled + 4.5 * base_penalty + solvent_waste - 4.0 * time_scaled
    return {"yield_score": float(yield_score), "waste_score": float(waste_score)}

## 🔁 qLogEHVI Campaign Loop

Suggestions are generated without mutating state, then explicitly appended. Each accepted experiment is marked observed with both objective values.

In [ ]:
run_records = []

while len(campaign.observed_data()) < TARGET_OBSERVED_ROWS:
    remaining = TARGET_OBSERVED_ROWS - len(campaign.observed_data())
    suggestions = campaign.suggest_next(batch_size=min(2, remaining))
    suggestions.to_csv(LATEST_SUGGESTIONS_PATH, index=False)
    campaign.append_suggestions(suggestions)

    for _, suggestion in suggestions.iterrows():
        objective_values = simulate_coupled_objectives(suggestion)
        campaign.mark_observed(
            row_id=str(suggestion["row_id"]),
            objective_values=objective_values,
        )
        run_records.append(
            {
                "row_id": suggestion["row_id"],
                "solvent": suggestion["solvent"],
                "yield_score": objective_values["yield_score"],
                "waste_score": objective_values["waste_score"],
            }
        )
    campaign.reload()

run_log = pd.DataFrame(run_records)
run_log.tail()

## 📈 Pareto And Hypervolume Results

The Pareto front is shown in user-facing objective units. Hypervolume is computed in maximisation model space after applying each objective direction.

In [ ]:
campaign.summary()

In [ ]:
campaign.pareto_front()[["row_id", "yield_score", "waste_score", "solvent"]]

In [ ]:
campaign.plot_pareto(save_path=PARETO_PLOT_PATH)

In [ ]:
campaign.plot_hypervolume(save_path=HYPERVOLUME_PLOT_PATH)

In [ ]:
report = campaign.report()
campaign.export_report(REPORT_PATH)
report.keys()